# Mid-circuit measurement with separated DSP classification lines

This notebook runs a single-qubit mid-circuit measurement experiment with hardware DSP classification.  It first uses one effective decision line by sending the same line parameters to both DSP decision functions, and checks that each repeated measurement pair is concentrated on `gg` and `ee`.

Set `system_id`, `muxes`, and `target_indices` in the setup cell for your hardware before running the calibration and measurement cells.

Then it separates the two parallel decision lines around the GMM state centers.  For each single capture, the two DSP decision-line bits are accepted only when they are `00` or `11`; DSP-line outputs `01` and `10` are removed from the denominator.  The two repeated measurement outcomes are counted separately after that per-capture DSP post-selection, so repeated-outcome `01` and `10` remain in the denominator for `P(gg)+P(ee)`.  Because the two lines are parallel, the between-line region usually appears as only one of raw DSP `01` or `10` for a fixed line order, but both entries are always displayed.  The plots show the GMM event distribution, the decision lines, the raw DSP line-bit distribution, and the repeated-measurement probabilities before and after DSP post-selection.


In [ ]:
import numpy as np

import qubex as qx
from qubex.contrib import (
    build_gmm_midpoint_band_line_param_maps,
    build_gmm_midpoint_line_param_maps,
    build_gmm_separated_line_param_maps,
    plot_gmm_events_with_lines,
    plot_raw_dsp_output_distribution,
    plot_repeated_diagonal_comparison,
    plot_repeated_dsp_summary,
    plot_repeated_line_distance_sweep,
    plot_single_readout_equivalent_comparison,
    plot_single_readout_equivalent_line_distance_sweep,
    single_readout_fidelity_from_repeated_diagonal,
    summarize_repeated_dsp_classification,
    summary_unconditional_diagonal_fraction,
)

In [ ]:
system_id = "YOUR_SYSTEM_ID"
muxes = [0]
target_indices = (0,)

exp = qx.Experiment(
    system_id=system_id,
    muxes=muxes,
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

exp.connect()
ctx = exp.ctx
print("Experiment context is ready:", type(ctx).__name__)

assert len(target_indices) == 1, "This example expects one target qubit."
assert len(exp.qubit_labels) > max(target_indices), (
    "Select a valid target index for this mux."
)

targets = [exp.qubit_labels[index] for index in target_indices]
readout_targets = [ctx.resolve_read_label(target) for target in targets]

list(zip(targets, readout_targets, strict=True))

In [ ]:
# exp.configure()


In [ ]:
exp.tool.print_target_frequencies(targets)

In [ ]:
# Keep the calibration steps explicit.  If a later step is still running, a new
# notebook cell such as `exp.ctx` will appear to hang because it is queued.
run_rabi_calibration = True
run_hpi_calibration = True
rebuild_classifier = True

rabi_time_range = np.arange(0, 401, 16)
rabi_shots = 1024
rabi_plot = False
classifier_shots = 10000

if run_rabi_calibration:
    print(
        f"Starting simultaneous Rabi calibration: {len(rabi_time_range)} points, {rabi_shots} shots"
    )
    rabi_result = exp.obtain_rabi_params(
        targets,
        time_range=rabi_time_range,
        n_shots=rabi_shots,
        plot=rabi_plot,
        simultaneous=True,
        enable_tqdm=True,
        store_params=True,
    )
    print("Rabi calibration returned")
else:
    rabi_result = None

if rabi_result is not None:
    print("Rabi fit summary")
    for target, data in rabi_result.data.items():
        param = data.rabi_param
        ok = np.isfinite(param.r2) and param.r2 >= 0.5
        status = "OK" if ok else "NG"
        print(
            f"  {target}: {status}, "
            f"r2={param.r2:.4f}, "
            f"frequency={param.frequency:.6g}, "
            f"amplitude={param.amplitude:.6g}"
        )

if run_hpi_calibration:
    print("Starting HPI calibration")
    hpi_result = exp.calibrate_hpi_pulse(targets, plot=True)
    print("HPI calibration returned")
else:
    hpi_result = None

if rebuild_classifier:
    print(f"Starting classifier rebuild: {classifier_shots} shots")
    classifier_result = exp.build_classifier(
        targets, n_states=2, n_shots=classifier_shots, plot=True
    )
    print("Classifier rebuild returned")
else:
    classifier_result = None

print("Rabi calibration:", "run" if run_rabi_calibration else "skipped")
print("HPI calibration:", "run" if run_hpi_calibration else "skipped")
print("Classifier rebuild:", "run" if rebuild_classifier else "skipped")
print(
    "If GMM centers/stddevs are missing later, set rebuild_classifier=True and rerun this cell."
)

In [ ]:
classifier_map = dict(getattr(ctx, "classifiers", {}))
print(
    "Classifiers:",
    {
        target: type(classifier_map[target]).__name__
        for target in targets
        if target in classifier_map
    },
)

In [ ]:
# One-line baseline: line0 and line1 intentionally use the same midpoint separator.
baseline_line0_by_readout, baseline_line1_by_readout = (
    build_gmm_midpoint_line_param_maps(
        ctx,
        targets,
    )
)

plot_gmm_events_with_lines(
    ctx,
    targets,
    baseline_line0_by_readout,
    baseline_line1_by_readout,
    title="Single-line baseline: line0 and line1 are identical",
)

In [ ]:
# Separated-line run: move each line inward from its GMM center by sigma_multiplier * sigma.
# These line params are in normalized qubex I/Q units for plotting; the backend scales c for e7awghal DSP units.
sigma_multiplier = 1.0
separated_line0_by_readout, separated_line1_by_readout = (
    build_gmm_separated_line_param_maps(
        ctx,
        targets,
        sigma_multiplier=sigma_multiplier,
    )
)

plot_gmm_events_with_lines(
    ctx,
    targets,
    separated_line0_by_readout,
    separated_line1_by_readout,
    title=f"Separated lines: {sigma_multiplier:g} sigma from each GMM center",
    sigma_multiplier=sigma_multiplier,
)

In [ ]:
wait_ns = 512
readout_duration = round(exp.readout_duration)
readout_pre_margin = round(exp.readout_pre_margin)
readout_post_margin = round(exp.readout_post_margin)

readout_pulses = {
    readout_target: exp.readout(
        readout_target,
        duration=readout_duration,
        pre_margin=readout_pre_margin,
        post_margin=readout_post_margin,
    )
    for readout_target in readout_targets
}

with qx.PulseSchedule([*targets, *readout_targets]) as sequence:
    for target in targets:
        sequence.add(target, exp.get_hpi_pulse(target))
    sequence.barrier()

    for readout_target in readout_targets:
        sequence.add(readout_target, readout_pulses[readout_target])
    sequence.barrier()

    for line in [*targets, *readout_targets]:
        sequence.add(line, qx.Blank(wait_ns))
    sequence.barrier()

    for readout_target in readout_targets:
        sequence.add(readout_target, readout_pulses[readout_target])

sequence

In [ ]:
sequence.plot()

In [ ]:
# Each repetition is a single-shot DSP-classified experiment.
n_analysis_shots = 10000
shot_interval = 150 * 1024


def run_dsp_classification(line0_by_readout, line1_by_readout):
    """Run one DSP-classified measurement with the given line maps."""
    return ctx.measurement.execute(
        schedule=sequence,
        n_shots=n_analysis_shots,
        shot_interval=shot_interval,
        shot_averaging=False,
        time_integration=True,
        state_classification=True,
        classification_source="gmm_linear",
        classification_line_param0=line0_by_readout,
        classification_line_param1=line1_by_readout,
        readout_duration=readout_duration,
        readout_pre_margin=readout_pre_margin,
        readout_post_margin=readout_post_margin,
        plot=False,
    )

## Baseline: one effective line

This run sends the same midpoint separator to both hardware decision functions.  The raw DSP output should therefore be dominated by `00` and `11`; the repeated measurement pairs for each qubit should be concentrated on `gg` and `ee`.


In [ ]:
baseline_result = run_dsp_classification(
    baseline_line0_by_readout,
    baseline_line1_by_readout,
)

{target: len(baseline_result.data[target]) for target in targets}

In [ ]:
# DSP raw-label summary and plotting helpers are imported from qubex.contrib.


In [ ]:
baseline_summary = summarize_repeated_dsp_classification(
    baseline_result,
    targets,
    title="Single-line baseline",
    reject_dsp_ambiguous=False,
)
plot_raw_dsp_output_distribution(
    baseline_result,
    targets,
    title="Single-line baseline: raw DSP outputs",
)
plot_repeated_dsp_summary(
    baseline_summary,
    targets,
    title="Single-line baseline: repeated-measurement probabilities",
)

## Separated lines with ambiguous-output rejection

This run separates the two decision lines.  For each capture, the two raw DSP line bits `00` and `11` are accepted and mapped to logical `g` and `e`; raw DSP line-bit outputs `01` and `10` sit between the two lines and are dropped.  The repeated measurement outcome is counted afterward, so repeated-outcome `01`/`10` are not dropped by this DSP-line post-selection.


In [ ]:
separated_result = run_dsp_classification(
    separated_line0_by_readout,
    separated_line1_by_readout,
)

{target: len(separated_result.data[target]) for target in targets}

In [ ]:
separated_summary = summarize_repeated_dsp_classification(
    separated_result,
    targets,
    title=f"Separated lines with {sigma_multiplier:g}-sigma rejection band",
    reject_dsp_ambiguous=True,
)
plot_raw_dsp_output_distribution(
    separated_result,
    targets,
    title="Separated lines: raw DSP outputs",
)
plot_repeated_dsp_summary(
    separated_summary,
    targets,
    title="Separated lines: post-selected repeated-measurement probabilities",
)

In [ ]:
# Overall aggregation is intentionally omitted; the summaries below are evaluated per qubit.


In [ ]:
print("Per-qubit comparison")
for target in targets:
    baseline = baseline_summary[target]
    separated = separated_summary[target]
    print(target)
    print(f"  baseline P(gg)+P(ee): {baseline['diagonal_probability']:.4f}")
    print(f"  separated retained fraction: {separated['retained_fraction']:.4f}")
    print(f"  separated P(gg)+P(ee): {separated['diagonal_probability']:.4f}")
    print(f"  separated P(gg): {separated['gg_probability']:.4f}")
    print(f"  separated P(ee): {separated['ee_probability']:.4f}")

## Effective repeated-measurement comparison

The next plots first compare the conditional repeated-measurement diagonal probability, `P(00)+P(11)`, and then convert it to a single-readout-equivalent fidelity using `P_same = f**2 + (1 - f)**2`.  The per-readout retained fraction is computed from each capture's raw DSP `00/11` survival rate, not from the two-capture pair denominator.  The sweep uses line distance from the midpoint separator: distance `0` is the one-line baseline, and larger distances reject more events near the decision boundary.


In [ ]:
plot_repeated_diagonal_comparison(
    baseline_summary,
    separated_summary,
    targets=targets,
)
plot_single_readout_equivalent_comparison(
    baseline_summary,
    separated_summary,
    targets=targets,
)

## Line-distance sweep

This sweep opens a symmetric rejection band around the midpoint separator and evaluates each qubit separately.  It reuses `baseline_result` for distance `0` and acquires new DSP-classified data for nonzero distances.  The first sweep plot shows the two-capture repeated-measurement quantity, and the second shows the single-readout-equivalent fidelity and per-capture retained fraction.


In [ ]:
# Line-distance sweep helpers are imported from qubex.contrib.


In [ ]:
line_distance_fractions = np.linspace(0.0, 0.8, 9)
line_distance_sweep_records = []
line_distance_sweep_results = {}

for distance_fraction in line_distance_fractions:
    line0_by_readout, line1_by_readout, line_distances = (
        build_gmm_midpoint_band_line_param_maps(
            ctx,
            targets,
            distance_fraction=distance_fraction,
        )
    )
    if np.isclose(distance_fraction, 0.0):
        result = baseline_result
    else:
        result = run_dsp_classification(line0_by_readout, line1_by_readout)
    line_distance_sweep_results[float(distance_fraction)] = result

    summaries = summarize_repeated_dsp_classification(
        result,
        targets,
        reject_dsp_ambiguous=True,
        verbose=False,
    )

    for target in targets:
        summary = summaries[target]
        line_distance_sweep_records.append(
            {
                "target": target,
                "line_distance_fraction": float(distance_fraction),
                "line_distance": float(line_distances[target]),
                "retained_fraction": float(summary["retained_fraction"]),
                "conditional_fidelity": float(summary["diagonal_probability"]),
                "single_readout_fidelity": float(
                    single_readout_fidelity_from_repeated_diagonal(
                        summary["diagonal_probability"]
                    )
                ),
                "single_readout_retained_fraction": float(
                    summary["single_readout_retained_fraction"]
                ),
                "all_shot_diagonal_fraction": float(
                    summary_unconditional_diagonal_fraction(summary)
                ),
                "rejected_total": int(summary["dsp_line_rejected_total"]),
            }
        )
line_distance_sweep_records[:3]

In [ ]:
plot_repeated_line_distance_sweep(
    line_distance_sweep_records,
    targets=targets,
)
plot_single_readout_equivalent_line_distance_sweep(
    line_distance_sweep_records,
    targets=targets,
)